# 🏆 World Cup Monte Carlo Simulation

Welcome to the **Complete Tournament Simulator**! This notebook leverages Monte Carlo methods to simulate the entire World Cup (Group Stage followed by the complete 32-team Knockout Bracket) up to 10,000 times.

### Why Monte Carlo Simulation?
Football is notoriously unpredictable. While a single prediction gives us a baseline, a Monte Carlo simulation (running the tournament thousands of times with probabilistic results) gives us the *true likelihood* of various outcomes: qualifying from the group stage, reaching the quarterfinals, or lifting the trophy.

In [ ]:
import os
import itertools
import pandas as pd
import numpy as np
import joblib
from collections import defaultdict
from tqdm.notebook import tqdm
from IPython.display import display, HTML

# Setup
model_path = os.path.join('..', 'models', 'xgboost_wm_modelV4.joblib')
master_path = os.path.join('..', 'data', 'processed', 'features', 'MASTER_dataset.csv')

if not os.path.exists(model_path) or not os.path.exists(master_path):
    raise FileNotFoundError("Please run the ML pipeline ('python run_pipeline.py all') first to compile model and master datasets.")

model = joblib.load(model_path)
master_df = pd.read_csv(master_path)

groups = {
    'A': ['Mexico', 'South Africa', 'South Korea', 'Czech Republic'],
    'B': ['Canada', 'Bosnia and Herzegovina', 'Qatar', 'Switzerland'],
    'C': ['Brazil', 'Morocco', 'Haiti', 'Scotland'],
    'D': ['USA', 'Paraguay', 'Australia', 'Turkey'],
    'E': ['Germany', 'Curacao', 'Ivory Coast', 'Ecuador'],
    'F': ['Netherlands', 'Japan', 'Sweden', 'Tunisia'],
    'G': ['Belgium', 'Egypt', 'Iran', 'New Zealand'],
    'H': ['Spain', 'Cape Verde', 'Saudi Arabia', 'Uruguay'],
    'I': ['France', 'Senegal', 'Iraq', 'Norway'],
    'J': ['Argentina', 'Algeria', 'Austria', 'Jordan'],
    'K': ['Portugal', 'DR Congo', 'Uzbekistan', 'Colombia'],
    'L': ['England', 'Croatia', 'Ghana', 'Panama']
}

features = [
    'Delta_Total_Market_Value', 'Delta_Median_Top11_Value', 'Delta_Chemistry',
    'Delta_Form_Rating', 'Delta_UCL_Minutes', 'Delta_Tournament_Minutes',
    'Delta_Average_Age', 'Delta_TM_Value_Rank', 'Delta_FIFA_Rank', 'Delta_FIFA_Points',
    'Is_Neutral'
]

all_teams = [team for group_teams in groups.values() for team in group_teams]
print(f"✓ Initialized. Total teams loaded: {len(all_teams)}")

## 1. Pre-calculate Pairwise Matchup Probabilities

To achieve high-speed simulations, we pre-evaluate our XGBoost model on all possible 1,128 team combinations. This allows our simulation loop to fetch probabilities via high-speed dictionary lookups instead of executing model inference repeatedly.

In [ ]:
print("🧠 Pre-calculating all possible 1,128 matchup outcome probabilities...")
match_probs = {}
team_fifa_points = {}

for team in all_teams:
    team_data = master_df[master_df['Country'].str.lower() == team.lower()]
    team_fifa_points[team] = team_data.iloc[0]['FIFA_Points']

for team_a, team_b in itertools.combinations(all_teams, 2):
    data_a = master_df[master_df['Country'].str.lower() == team_a.lower()].iloc[0]
    data_b = master_df[master_df['Country'].str.lower() == team_b.lower()].iloc[0]
    
    delta_dict = {
        'Delta_Total_Market_Value': data_a['Total_Market_Value_mEUR'] - data_b['Total_Market_Value_mEUR'],
        'Delta_Median_Top11_Value': data_a['Median_Top11_Market_Value_mEUR'] - data_b['Median_Top11_Market_Value_mEUR'],
        'Delta_Chemistry': data_a['Chemistry_Score'] - data_b['Chemistry_Score'],
        'Delta_Form_Rating': data_a['Current_Form_Rating'] - data_b['Current_Form_Rating'],
        'Delta_UCL_Minutes': data_a['Total_UCL_Minutes'] - data_b['Total_UCL_Minutes'],
        'Delta_Tournament_Minutes': data_a['Total_Tournament_Minutes'] - data_b['Total_Tournament_Minutes'],
        'Delta_Average_Age': data_a['Average_Age'] - data_b['Average_Age'],
        'Delta_TM_Value_Rank': data_a['TM_Value_Rank'] - data_b['TM_Value_Rank'],
        'Delta_FIFA_Rank': data_a['FIFA_Rank'] - data_b['FIFA_Rank'],
        'Delta_FIFA_Points': data_a['FIFA_Points'] - data_b['FIFA_Points'],
        'Is_Neutral': 1
    }
    X_pred = pd.DataFrame([delta_dict])[features]
    probs = model.predict_proba(X_pred)[0]
    match_probs[(team_a, team_b)] = probs
print("✓ Matchup database compiled successfully!")

## 2. Define Simulation Helper Functions

In the knockout stages, matches cannot end in a draw. We implement a knockout game simulator that normalizes the probability of Team A winning versus Team B winning, discarding the draw probability.

In [ ]:
def play_ko_match(team_a, team_b):
    """Simulates a knockout game where a draw is impossible (normalizing probabilities)."""
    if (team_a, team_b) in match_probs:
        probs = match_probs[(team_a, team_b)]
        p_a = probs[2] / (probs[2] + probs[0]) # Normalize Home Win vs Away Win
        return team_a if np.random.rand() < p_a else team_b
    else:
        probs = match_probs[(team_b, team_a)]
        p_b = probs[2] / (probs[2] + probs[0])
        return team_b if np.random.rand() < p_b else team_a

def play_round(teams):
    """Plays a complete KO round by matching pairs of consecutive teams in the list."""
    return [play_ko_match(teams[i], teams[i+1]) for i in range(0, len(teams), 2)]

## 3. Run Monte Carlo Simulations

We run 10,000 complete World Cup simulations. For each simulation, we track standings, qualify group leaders, determine the 8 best 3rd placed teams, construct the Round of 32, and play through the brackets.

In [ ]:
simulations = 10000

# Track standings stats
group_stats = defaultdict(lambda: {'1st': 0, '2nd': 0, '3rd': 0, '4th': 0, 'Advances_Group': 0})
# Track knockout round progression
ko_stats = defaultdict(lambda: {'R32': 0, 'R16': 0, 'QF': 0, 'SF': 0, 'Final': 0, 'Win': 0})

for _ in tqdm(range(simulations), desc="Simulating World Cups"):
    winners, runners_up, thirds = {}, {}, []
    
    # --- 1. GROUP PHASE ---
    for group_name, teams in groups.items():
        standings = {team: 0 for team in teams}
        for team_a, team_b in itertools.combinations(teams, 2):
            probs = match_probs[(team_a, team_b)]
            outcome = np.random.choice([0, 1, 2], p=probs)
            if outcome == 2: standings[team_a] += 3
            elif outcome == 0: standings[team_b] += 3
            else:
                standings[team_a] += 1
                standings[team_b] += 1
                
        sorted_group = sorted(standings.items(), key=lambda x: (x[1], team_fifa_points[x[0]]), reverse=True)
        t1, t2, t3, t4 = [t[0] for t in sorted_group]
        
        group_stats[t1]['1st'] += 1
        group_stats[t2]['2nd'] += 1
        group_stats[t3]['3rd'] += 1
        group_stats[t4]['4th'] += 1
        
        group_stats[t1]['Advances_Group'] += 1
        group_stats[t2]['Advances_Group'] += 1
        
        winners[group_name] = t1
        runners_up[group_name] = t2
        thirds.append((t3, sorted_group[2][1], team_fifa_points[t3]))
        
    # Select 8 best 3rd-placed teams
    best_thirds = [t[0] for t in sorted(thirds, key=lambda x: (x[1], x[2]), reverse=True)[:8]]
    for t3_team in best_thirds:
        group_stats[t3_team]['Advances_Group'] += 1
    
    # --- 2. ROUND OF 32 BRACKET ---
    r32_teams = [
        winners['A'], best_thirds[0], winners['B'], best_thirds[1],
        winners['C'], best_thirds[2], winners['D'], best_thirds[3],
        winners['E'], best_thirds[4], winners['F'], best_thirds[5],
        winners['G'], best_thirds[6], winners['H'], best_thirds[7],
        winners['I'], runners_up['A'], winners['J'], runners_up['B'],
        winners['K'], runners_up['C'], winners['L'], runners_up['D'],
        runners_up['E'], runners_up['F'], runners_up['G'], runners_up['H'],
        runners_up['I'], runners_up['J'], runners_up['K'], runners_up['L']
    ]
    for t in r32_teams: ko_stats[t]['R32'] += 1
        
    # --- 3. BRACKET ELIMINATIONS ---
    r16_teams = play_round(r32_teams)
    for t in r16_teams: ko_stats[t]['R16'] += 1
        
    qf_teams = play_round(r16_teams)
    for t in qf_teams: ko_stats[t]['QF'] += 1
        
    sf_teams = play_round(qf_teams)
    for t in sf_teams: ko_stats[t]['SF'] += 1
        
    final_teams = play_round(sf_teams)
    for t in final_teams: ko_stats[t]['Final'] += 1
        
    champion = play_round(final_teams)[0]
    ko_stats[champion]['Win'] += 1

## 4. Visualizing Results

### Complete Tournament Prognosis Summary

We compile the knockout bracket statistics into a beautifully formatted Pandas dataframe styled with color gradients.

In [ ]:
results = []
for team, t_stats in ko_stats.items():
    results.append({
        'Team': team,
        'Round of 32': (t_stats['R32'] / simulations) * 100,
        'Round of 16': (t_stats['R16'] / simulations) * 100,
        'Quarterfinals': (t_stats['QF'] / simulations) * 100,
        'Semifinals': (t_stats['SF'] / simulations) * 100,
        'Finals': (t_stats['Final'] / simulations) * 100,
        'CHAMPION': (t_stats['Win'] / simulations) * 100
    })

df_results = pd.DataFrame(results).sort_values(by='CHAMPION', ascending=False).reset_index(drop=True)
format_dict = {col: '{:.1f}%' for col in df_results.columns if col != 'Team'}

display(HTML("<h2>🏆 Monte Carlo World Cup Prognosis (10,000 Runs)</h2>"))
df_results.head(15).style.format(format_dict).set_properties(**{'text-align': 'center'}).background_gradient(subset=['CHAMPION'], cmap='Greens')